# Unidad 7: Lectura y escritura de datos

**Tecnicatura en Datos – Apache Spark (Databricks)**  
Notebook de práctica — `spark` ya está disponible en el entorno

---

## Contenido

1. CSV: opciones de lectura y schema explícito
2. JSON: datos planos y anidados
3. Parquet: columnar, compresión y partition pruning
4. Delta Lake: ACID, time travel y MERGE
5. JDBC: conexión a bases de datos relacionales
6. Opciones de escritura (`saveMode`)
7. Práctica guiada

## Configuración del entorno

> En Databricks `spark` ya está disponible. Este bloque es solo para validar la sesión.

In [ ]:
# En Databricks: spark ya está disponible
sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"App name:      {sc.appName}")

---
## 1. Lectura de CSV

CSV es el formato más simple y universal. Sus principales limitaciones en producción:
- No guarda tipos de datos → requiere `inferSchema` (lento) o schema explícito
- No soporta compresión eficiente columnar
- No es splittable sin compresión

### Ejemplo 1 — Simple: crear datos y guardar como CSV

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import col

# Crear datos de ejemplo
data_ventas = [
    (1, "Ana",   1500.0, "AR", "2024-01-10"),
    (2, "Luis",   800.0, "MX", "2024-01-11"),
    (3, "Marta", 2200.0, "CL", "2024-01-12"),
    (4, "Pedro",  None,  "AR", "2024-01-13"),   # Fila con nulo
    (5, "Sofía",  600.0, "BR", "2024-01-14"),
]

schema_ventas = StructType([
    StructField("id",      IntegerType(), False),
    StructField("cliente", StringType(),  True),
    StructField("monto",   DoubleType(),  True),
    StructField("pais",    StringType(),  True),
    StructField("fecha",   StringType(),  True),
])

df_ventas = spark.createDataFrame(data_ventas, schema_ventas)

# Guardar como CSV (para simular una fuente de datos)
df_ventas.write.mode("overwrite").option("header", "true").csv("/tmp/u07_ventas_csv/")

# Leer de vuelta con inferSchema
df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv("/tmp/u07_ventas_csv/")
df_csv.printSchema()
df_csv.show()

**Resultado esperado:**
```
root
 |-- id: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- pais: string (nullable = true)
 |-- fecha: string (nullable = true)

+---+-------+------+----+----------+
| id|cliente| monto|pais|     fecha|
+---+-------+------+----+----------+
|  1|    Ana|1500.0|  AR|2024-01-10|
|  2|   Luis| 800.0|  MX|2024-01-11|
|  3|  Marta|2200.0|  CL|2024-01-12|
|  4|  Pedro|  null|  AR|2024-01-13|
|  5|  Sofía| 600.0|  BR|2024-01-14|
+---+-------+------+----+----------+
```

### Ejemplo 2 — Medio: schema explícito con opciones avanzadas

In [ ]:
schema_explicito = StructType([
    StructField("id",      IntegerType(), False),
    StructField("cliente", StringType(),  True),
    StructField("monto",   DoubleType(),  True),
    StructField("pais",    StringType(),  True),
    StructField("fecha",   StringType(),  True),
])

df_schema = (spark.read
    .schema(schema_explicito)                 # schema explícito: más rápido y seguro
    .option("header", "true")
    .option("nullValue", "")                  # interpretar vacíos como null
    .option("mode", "DROPMALFORMED")          # descartar filas con errores de tipo
    .csv("/tmp/u07_ventas_csv/")
)

print(f"Filas: {df_schema.count()}")
df_schema.show()

# Comparar modos:
print("\nModos disponibles:")
print("  PERMISSIVE   → null en campos con error (default)")
print("  DROPMALFORMED → descarta filas con error")
print("  FAILFAST      → lanza excepción al primer error")

**Resultado esperado:**
```
Filas: 5
+---+-------+------+----+----------+
| id|cliente| monto|pais|     fecha|
+---+-------+------+----+----------+
|  1|    Ana|1500.0|  AR|2024-01-10|
...
```

### Ejemplo 3 — Avanzado: metadata de origen con `input_file_name()`

In [ ]:
from pyspark.sql.functions import input_file_name, regexp_extract

df_con_origen = (spark.read
    .schema(schema_explicito)
    .option("header", "true")
    .csv("/tmp/u07_ventas_csv/")
    .withColumn("archivo_origen", input_file_name())
    # Extraer solo el nombre del archivo de la ruta completa
    .withColumn("nombre_archivo", regexp_extract(col("archivo_origen"), r"([^/]+)\.csv", 1))
)

df_con_origen.select("id", "cliente", "monto", "nombre_archivo").show(truncate=False)

**Resultado esperado:**
```
+---+-------+------+--------------------+
| id|cliente| monto|      nombre_archivo|
+---+-------+------+--------------------+
|  1|    Ana|1500.0|part-00000-...      |
|  2|   Luis| 800.0|part-00000-...      |
...
```
> `input_file_name()` devuelve la ruta completa del archivo que originó cada fila. Esencial para trazabilidad en pipelines de ingesta.

---
## 2. Parquet: el formato de producción

Parquet almacena datos **por columna** en lugar de por fila. Ventajas:
- Compresión nativa muy eficiente (columnas del mismo tipo comprimen mejor)
- **Predicado pushdown**: si solo necesitás una columna, Spark no lee las demás
- Preserva tipos de datos en metadatos (no necesitás schema al leer)
- Splittable para lectura paralela

### Ejemplo 1 — Simple: leer y escribir Parquet

In [ ]:
# Escribir como Parquet
df_ventas.write.mode("overwrite").parquet("/tmp/u07_ventas_parquet/")

# Leer — el schema se recupera automáticamente de los metadatos
df_parquet = spark.read.parquet("/tmp/u07_ventas_parquet/")

print("Schema desde metadatos Parquet (sin inferSchema):")
df_parquet.printSchema()
df_parquet.show()

**Resultado esperado:**
```
Schema desde metadatos Parquet (sin inferSchema):
root
 |-- id: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- pais: string (nullable = true)
 |-- fecha: string (nullable = true)

+---+-------+------+----+----------+
| id|cliente| monto|pais|     fecha|
...                                   ← tipos preservados exactamente
```

### Ejemplo 2 — Medio: escritura particionada con `partitionBy`

In [ ]:
from pyspark.sql.functions import lit

# Agregar columnas de partición
df_con_particion = df_ventas \
    .withColumn("año", lit(2024)) \
    .withColumn("mes", lit(1))

# Escritura particionada
df_con_particion.write \
    .mode("overwrite") \
    .partitionBy("año", "mes", "pais") \
    .parquet("/tmp/u07_ventas_particionado/")

print("Estructura de particiones creada")

# Leer con filtro — Spark solo lee la partición necesaria (partition pruning)
df_ar = spark.read.parquet("/tmp/u07_ventas_particionado/") \
    .filter("año = 2024 AND pais = 'AR'")

print(f"\nFilas de Argentina en 2024: {df_ar.count()}")
df_ar.show()

# Ver el plan — notar PartitionFilters
print("\nPlan físico (buscar PartitionFilters):")
df_ar.explain("formatted")

**Resultado esperado:**
```
Filas de Argentina en 2024: 2
+---+-------+------+----+----------+----+---+
| id|cliente| monto|pais|     fecha| año|mes|
+---+-------+------+----+----------+----+---+
|  1|    Ana|1500.0|  AR|2024-01-10|2024|  1|
|  4|  Pedro|  null|  AR|2024-01-13|2024|  1|
+---+-------+------+----+----------+----+---+

Plan físico:
  PartitionFilters: [isnotnull(año), (año = 2024), isnotnull(pais), (pais = AR)]
```
> El `PartitionFilters` en el plan significa que Spark **no escanea** las otras particiones.

### Ejemplo 3 — Avanzado: comparar rendimiento CSV vs Parquet

In [ ]:
import time
from pyspark.sql.functions import rand, when

# Generar dataset mediano
df_grande = (spark.range(200_000)
    .withColumn("cliente", (col("id") % 1000).cast("string"))
    .withColumn("monto",   (rand() * 5000).cast("double"))
    .withColumn("pais",    when(col("id") % 3 == 0, "AR")
                           .when(col("id") % 3 == 1, "MX")
                           .otherwise("CL"))
)

# Guardar en ambos formatos
df_grande.write.mode("overwrite").option("header","true").csv("/tmp/u07_grande_csv/")
df_grande.write.mode("overwrite").parquet("/tmp/u07_grande_parquet/")

# Benchmark: solo leer columna monto y calcular promedio
t0 = time.time()
spark.read.option("header","true").option("inferSchema","true") \
    .csv("/tmp/u07_grande_csv/").agg({"monto": "avg"}).collect()
t_csv = time.time() - t0

t1 = time.time()
spark.read.parquet("/tmp/u07_grande_parquet/").agg({"monto": "avg"}).collect()
t_parquet = time.time() - t1

print(f"CSV     avg(monto): {t_csv:.2f}s")
print(f"Parquet avg(monto): {t_parquet:.2f}s")
print(f"Speedup: {t_csv/t_parquet:.1f}x más rápido con Parquet")

**Resultado esperado (valores orientativos según cluster):**
```
CSV     avg(monto): 4.82s
Parquet avg(monto): 0.63s
Speedup: 7.7x más rápido con Parquet
```
> La diferencia es mayor a mayor número de columnas porque Parquet solo lee las columnas pedidas.

---
## 3. Delta Lake: ACID + Time Travel

Delta Lake agrega sobre Parquet:
- **Transacciones ACID**: las escrituras son atómicas
- **Time Travel**: acceder a versiones anteriores
- **Schema enforcement**: rechaza datos con schema incorrecto
- **MERGE INTO**: upserts eficientes

### Ejemplo 1 — Simple: escribir y leer Delta

In [ ]:
# Escribir en formato Delta
df_ventas.write.format("delta").mode("overwrite").save("/tmp/u07_ventas_delta/")

# Leer
df_delta = spark.read.format("delta").load("/tmp/u07_ventas_delta/")
print("Lectura desde Delta:")
df_delta.show()

# Hacer una segunda escritura para generar versión 1
df_v2 = df_ventas.withColumn("monto", col("monto") * 1.1)   # aumentar 10%
df_v2.write.format("delta").mode("overwrite").save("/tmp/u07_ventas_delta/")

print("\nVersión actual (v1):")
spark.read.format("delta").load("/tmp/u07_ventas_delta/").show()

### Ejemplo 2 — Medio: Time Travel

In [ ]:
# Leer versión anterior (versión 0 = primera escritura)
df_version_0 = (spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load("/tmp/u07_ventas_delta/")
)

print("Versión 0 (montos originales):")
df_version_0.show()

# Comparar montos entre versiones
df_actual = spark.read.format("delta").load("/tmp/u07_ventas_delta/")
print("\nComparación versión 0 vs versión 1:")
df_version_0.select("id", col("monto").alias("monto_v0")) \
    .join(df_actual.select("id", col("monto").alias("monto_v1")), "id") \
    .show()

**Resultado esperado:**
```
Versión 0 (montos originales):
+---+-------+------+----+----------+
| id|cliente| monto|pais|     fecha|
+---+-------+------+----+----------+
|  1|    Ana|1500.0|  AR|2024-01-10|
...

Comparación versión 0 vs versión 1:
+---+--------+--------+
| id|monto_v0|monto_v1|
+---+--------+--------+
|  1|  1500.0|  1650.0|   ← +10%
|  2|   800.0|   880.0|
...
```

### Ejemplo 3 — Avanzado: UPSERT con MERGE INTO

In [ ]:
from delta.tables import DeltaTable

# Tabla destino (Delta existente)
tabla_delta = DeltaTable.forPath(spark, "/tmp/u07_ventas_delta/")

print("Estado antes del MERGE:")
tabla_delta.toDF().orderBy("id").show()

# Nuevos datos: actualizaciones + inserciones
df_nuevos = spark.createDataFrame([
    (1, "Ana",   3000.0, "AR", "2024-02-01"),   # actualización (id=1 existe)
    (6, "Diego", 1200.0, "MX", "2024-02-01"),   # inserción nueva (id=6 no existe)
], ["id", "cliente", "monto", "pais", "fecha"])

# MERGE
(tabla_delta.alias("destino")
    .merge(df_nuevos.alias("nuevos"), "destino.id = nuevos.id")
    .whenMatchedUpdate(set={
        "monto": "nuevos.monto",
        "fecha": "nuevos.fecha",
    })
    .whenNotMatchedInsert(values={
        "id":      "nuevos.id",
        "cliente": "nuevos.cliente",
        "monto":   "nuevos.monto",
        "pais":    "nuevos.pais",
        "fecha":   "nuevos.fecha",
    })
    .execute()
)

print("\nEstado después del MERGE:")
tabla_delta.toDF().orderBy("id").show()

**Resultado esperado:**
```
Estado antes del MERGE:
+---+-------+------+----+----------+
| id|cliente| monto|pais|     fecha|
|  1|    Ana|1650.0|  AR|2024-01-10|
|  2|   Luis| 880.0|  MX|2024-01-11|
...

Estado después del MERGE:
+---+-------+------+----+----------+
|  1|    Ana|3000.0|  AR|2024-02-01|  ← actualizado
|  2|   Luis| 880.0|  MX|2024-01-11|  ← sin cambios
...
|  6|  Diego|1200.0|  MX|2024-02-01|  ← insertado
```

---
## 4. Opciones de escritura (saveMode)

| Modo | Comportamiento |
|------|----------------|
| `overwrite` | Reemplaza todo el destino |
| `append` | Agrega al destino existente |
| `ignore` | No hace nada si el destino existe |
| `errorIfExists` | Lanza error si el destino existe (default) |

In [ ]:
df_pequeño = spark.createDataFrame([(10, "Extra", 999.0, "AR", "2024-02-01")],
                                   ["id", "cliente", "monto", "pais", "fecha"])

# Antes: 5 filas
print(f"Antes del append: {spark.read.parquet('/tmp/u07_ventas_parquet/').count()} filas")

# append: agrega sin borrar
df_pequeño.write.mode("append").parquet("/tmp/u07_ventas_parquet/")

print(f"Después del append: {spark.read.parquet('/tmp/u07_ventas_parquet/').count()} filas")

# ignore: no hace nada si ya existe
df_pequeño.write.mode("ignore").parquet("/tmp/u07_ventas_parquet/")
print(f"Después de ignore:  {spark.read.parquet('/tmp/u07_ventas_parquet/').count()} filas (sin cambio)")

# overwrite: reemplaza todo
df_ventas.write.mode("overwrite").parquet("/tmp/u07_ventas_parquet/")
print(f"Después de overwrite: {spark.read.parquet('/tmp/u07_ventas_parquet/').count()} filas (reiniciado)")

**Resultado esperado:**
```
Antes del append: 5 filas
Después del append: 6 filas
Después de ignore:  6 filas (sin cambio)
Después de overwrite: 5 filas (reiniciado)
```

---
## 5. Práctica guiada

### Ejercicio: pipeline completo CSV → Silver (Parquet particionado)

In [ ]:
from pyspark.sql.functions import col, when, current_timestamp, year, month

# --- GENERAR DATOS RAW (simula una fuente CSV real) ---
data_raw = [(i, f"cliente_{i%20}", float(i * 73.5) if i % 8 != 0 else None,
             ["AR","MX","CL","BR"][i%4], f"2024-{(i%12)+1:02d}-{(i%28)+1:02d}")
            for i in range(1, 201)]

df_raw = spark.createDataFrame(data_raw, ["id", "cliente", "monto", "pais", "fecha"])

# Guardar como CSV (fuente)
df_raw.write.mode("overwrite").option("header", "true").csv("/tmp/u07_practica_csv/")

# --- BRONZE: leer CSV con schema explícito ---
schema_raw = StructType([
    StructField("id",      IntegerType(), False),
    StructField("cliente", StringType(),  True),
    StructField("monto",   DoubleType(),  True),
    StructField("pais",    StringType(),  True),
    StructField("fecha",   StringType(),  True),
])

df_bronze = spark.read.schema(schema_raw).option("header", "true") \
    .csv("/tmp/u07_practica_csv/")
print(f"Bronze: {df_bronze.count()} filas leídas")

# --- SILVER: limpiar y enriquecer ---
df_silver = (df_bronze
    .filter(col("monto").isNotNull() & (col("monto") > 0))
    .withColumn("categoria",
        when(col("monto") < 2000, "bajo")
        .when(col("monto") < 8000, "medio")
        .otherwise("alto"))
    .withColumn("procesado_en", current_timestamp())
    .withColumn("año", year(col("fecha").cast("date")))
    .withColumn("mes", month(col("fecha").cast("date")))
)

invalidas = df_bronze.count() - df_silver.count()
print(f"Silver: {df_silver.count()} filas válidas, {invalidas} descartadas")

# --- GUARDAR SILVER como Parquet particionado ---
df_silver.write \
    .mode("overwrite") \
    .partitionBy("año", "mes", "pais") \
    .parquet("/tmp/u07_practica_silver/")

# --- VERIFICAR con filtro (partition pruning) ---
df_verify = spark.read.parquet("/tmp/u07_practica_silver/") \
    .filter("pais = 'AR' AND mes = 1")
print(f"\nFilas AR, mes 1: {df_verify.count()}")
df_verify.select("id", "cliente", "monto", "categoria").show(10)

**Resultado esperado:**
```
Bronze: 200 filas leídas
Silver: 175 filas válidas, 25 descartadas

Filas AR, mes 1: 4
+---+----------+------+---------+
| id|   cliente| monto|categoria|
+---+----------+------+---------+
|  4|cliente_4 | 294.0|     bajo|
...
```

---
## Preguntas de revisión

1. ¿Por qué Parquet es más eficiente que CSV para análisis analíticos?
2. ¿Cuál es la diferencia entre `repartition` y `partitionBy` al escribir?
3. ¿Qué es el partition pruning y cómo lo aprovecha Spark?
4. ¿Qué agrega Delta Lake sobre Parquet? ¿Cuándo usarlo?
5. ¿Qué riesgo tiene usar `inferSchema=True` en producción?
6. ¿Qué modo de escritura usarías para un pipeline de carga diaria incremental?